In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

In [4]:
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

In [5]:
X_train = pd.read_csv("../../data/processed/X_train.csv")
X_test = pd.read_csv("../../data/processed/X_test.csv")
y_train = pd.read_csv("../../data/processed/y_train.csv", header=None).squeeze()
y_test = pd.read_csv("../../data/processed/y_test.csv", header=None).squeeze()

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler


def make_preprocessor(X):
    """Build preprocessor from training data only (avoids data leakage)."""
    top_5_cities = X["city"].value_counts().head(5).index.tolist()
    encoder = OneHotEncoder(
        categories=[top_5_cities], handle_unknown="ignore", sparse_output=False
    )
    numerical_columns = X.drop(columns=["city"]).columns
    return ColumnTransformer(
        transformers=[
            ("city", encoder, ["city"]),
            ("num", StandardScaler(), numerical_columns),
        ]
    )

In [ ]:
base_models = {
    "LinearRegression": Pipeline([
        ("preprocessor", make_preprocessor(X_train)),
        ("model", LinearRegression()),
    ]),
    "Ridge": Pipeline([
        ("preprocessor", make_preprocessor(X_train)),
        ("model", Ridge(alpha=1.0)),
    ]),
    "Lasso": Pipeline([
        ("preprocessor", make_preprocessor(X_train)),
        ("model", Lasso(alpha=1.0)),
    ]),
    "RandomForestRegressor": Pipeline([
        ("preprocessor", make_preprocessor(X_train)),
        ("model", RandomForestRegressor(random_state=42)),
    ]),
    "XGBRegressor": Pipeline([
        ("preprocessor", make_preprocessor(X_train)),
        ("model", XGBRegressor(random_state=42)),
    ]),
}

In [7]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

results = []
for model_name, model in base_models.items():
     model.fit(X_train, y_train)
     y_pred_model = model.predict(X_test)
     mse = mean_squared_error(y_test, y_pred_model)
     mae = mean_absolute_error(y_test, y_pred_model)
     r2 = r2_score(y_test, y_pred_model)
     results.append({
          "Name" : model_name,
          "MSE" : mse,
          "MAE" : mae,
          "r2_score" : r2
          })

In [8]:
experiment_results = pd.DataFrame(results).round(2)
experiment_results

,Name,MSE,MAE,r2_score
0,LinearRegression,2.418686e+10,116153.01,0.66
1,Ridge,2.418858e+10,116161.60,0.66
2,Lasso,2.418696e+10,116153.20,0.66
3,RandomForestRegressor,2.574532e+10,118511.86,0.64
4,XGBRegressor,2.577827e+10,117118.01,0.64


In [10]:
import os 
os.makedirs('../../reports', exist_ok=True)

# Save Experiment results
experiment_results.to_csv('../../reports/01_experiment_results.csv', index=False)
print("Results saved to reports/01_experiment_results.csv")

Results saved to reports/01_experiment_results.csv
